In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import altair as alt
import pyarrow

In [2]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [2]:
df_hora = pd.read_csv('data/dados_hora.csv')
df_diarios = pd.read_csv('data/dados_diarios.csv')

In [7]:
df_hora['Resto (kWh)'] = df_hora['Rede Distribuição (kWh)'] - (df_hora['Eólica (kWh)'] + df_hora['Fotovoltaica (kWh)'] + df_hora['Hídrica (kWh)'] )

In [5]:
df_hora.describe()

,Cogeração (kWh),Eólica (kWh),Fotovoltaica (kWh),Hídrica (kWh),Outras Tecnologias (kWh),Rede Distribuição (kWh),Baixa Tensão (kWh),Média Tensão (kWh),Alta Tensão (kWh),Muito Alta Tensão (kWh),Dia,Mês,Ano,Mercado (kWh),Regime Especial (kWh),Total (kWh) (Consumido),Total (kWh) (Produzido),Resto (kWh)
count,109981.000000,1.099810e+05,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,1.099810e+05,1.099810e+05,109981.000000
mean,42107.022813,3.839374e+05,9399.109369,23595.893400,65396.774071,5.244362e+05,7.622991e+05,452311.478722,197598.470101,71292.390699,15.661051,6.298706,2024.084687,9.604390e+05,5.244274e+05,1.483501e+06,1.484866e+06,107503.796884
std,25470.863331,2.830175e+05,13271.313071,17321.372333,23767.494822,2.832140e+05,2.141918e+05,120512.717902,19256.419943,17242.062001,8.804726,3.533235,0.895114,3.616615e+05,2.832203e+05,2.786397e+05,2.783659e+05,26403.338227
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,2023.000000,-2.267620e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,19103.750000,1.496928e+05,0.250000,6168.500000,43854.468000,2.944831e+05,6.084690e+05,350009.521600,187234.566600,59152.361900,8.000000,3.000000,2023.000000,7.246130e+05,2.944607e+05,1.252330e+06,1.253547e+06,90500.750000
50%,47480.750000,3.193232e+05,183.250000,21362.000000,62524.546000,4.612802e+05,7.273177e+05,432361.067400,199737.935800,72896.314800,16.000000,6.000000,2024.000000,9.711140e+05,4.612802e+05,1.473203e+06,1.474199e+06,106883.789000
75%,63679.250000,5.651491e+05,18112.750000,40631.250000,83214.500000,7.011767e+05,8.786290e+05,554676.295600,210840.052400,85661.813000,23.000000,9.000000,2025.000000,1.214783e+06,7.011767e+05,1.668029e+06,1.669938e+06,123777.750000
max,102841.000000,1.238478e+06,50510.500000,95408.000000,144804.805006,1.466503e+06,1.729600e+06,761619.116200,244902.224000,109840.462400,31.000000,12.000000,2026.000000,2.178261e+06,1.466503e+06,2.561410e+06,2.561410e+06,201650.053023


In [6]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

# 2. "Derreter" o DataFrame
df_long = df_hora.melt(
    id_vars=['Data/Hora'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='kWh'
)

# 3. Criar o gráfico
chart = alt.Chart(df_long).mark_area().encode(
    alt.X('yearmonth(Data/Hora):T').axis(format='%b %Y', title='Mês/Ano'),
    alt.Y('sum(kWh):Q').title('Produção Total (kWh)'),
    
    # Adicionamos o 'sort' aqui para ordenar pela soma de kWh
    alt.Color('Tecnologia:N').scale(scheme='inferno').sort(
        alt.EncodingSortField(field='kWh', op='sum', order='descending')
    ),
    
    alt.Order('sum(kWh):Q', sort='descending'),
    
    tooltip=['yearmonth(Data/Hora)', 'Tecnologia', 'sum(kWh)']
).properties(
    width=800,
    height=400,
    title='Evolução Mensal da Produção de Energia'
).interactive()

chart.show()

alt.Chart(...)

In [ ]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1
))
r = "Eólica (kWh)"
anos_legend[-1] = "2026"
fig = px.bar_polar(
    dfII,
    r= r,
    theta="Mes/Ano",
    color="Ano",
    labels="Ano",
    title=f"Evolução Mensal da Produção de Energia: {r}",
    color_discrete_sequence=px.colors.qualitative.D3
).update_layout(
    showlegend=True,
    coloraxis_showscale=False,
    legend=dict(
        title="Ano de Produção",
        font=dict(size=12),
        # Isto garante que a legenda não fica preta se o fundo for escuro
        itemsizing='constant' 
    ),
    polar=dict(hole = 0.2,
          angularaxis=dict(
                type="category",
                # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                # O período tem de ser o número total de fatias para fechar o círculo
                period=len(dfII),
                tickvals=dfII['Mes/Ano'].tolist(),
                # O texto é que leva a lista com vazios
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=18, family='Arial', style='italic')
        ),  
        radialaxis=dict(
            showticklabels=True, 
            range=[0, dfII[r].max()*1.02],
            nticks=5, 
            tickfont=dict(size=18, family='Arial'),
            linecolor='black', 
            linewidth=1,
            layer='above traces', # Mudei para 'below' para as barras não taparem os números
            gridcolor='lightgrey'
        ),   
    ),
    
    height=600,
    width=800,
    margin=dict(b=30, t=80, l=0, r=0),
    )
fig.show()


    

In [31]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

# O melt mantém 'Ano' e 'Mês' e transforma as colunas de tecnologia em linhas
df_new = df_hora.melt(
    id_vars=['Ano', 'Mês'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='Producao'
)

dfIII = df_new.groupby(by=[df_new['Ano'],df_new['Mês'],df_new['Tecnologia']]).sum().reset_index()

dfIII['Ano'] = dfIII['Ano'].astype(str)
dfIII['Mes/Ano'] = dfIII['Mês'].astype(str) + '/' + dfIII['Ano'].astype(str)
r = "Producao"
meses = dict(zip(range(1,13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']))
anos_legend= list(dfIII.apply(
    lambda x: meses[x['Mês']]+ "/" + x['Ano'] if x['Mês'] %2 != 0 else "", axis=1
))
anos_legend[-1] = "2026"
fig = px.bar_polar(
    dfIII,
    r= r,
    theta="Mes/Ano",
    color="Tecnologia",
    labels="Ano",
    title=f"Evolução Mensal da Produção de Energia: {r}",
    color_discrete_sequence=px.colors.qualitative.Set3).update_layout(
    showlegend=True,
    coloraxis_showscale=False,
    legend=dict(
        title="Ano de Produção",
        font=dict(size=12),
        # Isto garante que a legenda não fica preta se o fundo for escuro
        itemsizing='constant' 
    ),
    polar=dict(hole = 0.2,
          angularaxis=dict(
                type="category",
                # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                categoryarray=dfIII['Mes/Ano'].tolist(),
                categoryorder="array",
                # O período tem de ser o número total de fatias para fechar o círculo
                period=len(dfIII['Mes/Ano'].unique()),
                tickvals=dfIII['Mes/Ano'].tolist(),
                # O texto é que leva a lista com vazios
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=18, family='Arial', style='italic')
        ),  
        radialaxis=dict(
            showticklabels=True, 
            range=[0, dfIII[r].max()*1.02],
            nticks=5, 
            tickfont=dict(size=18, family='Arial'),
            linecolor='black', 
            linewidth=1,
            layer='above traces', # Mudei para 'below' para as barras não taparem os números
            gridcolor='lightgrey'
        ),   
    ),
    
    height=1000,
    width=1000,
    margin=dict(b=30, t=80, l=100, r=60),
    )

fig.show()